**Algorithimic testing on roadNet-CA datasets**

In [ ]:
# Importing packages and files
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import seaborn as sns
from tqdm import tqdm

from Greedy_SOSP import Greedy_SOSP

DATA_PATH = Path("data") / "roadNet-CA.txt"
RESULTS_GSOSP_CSV = Path("roadNet-CA_results_GSOSP.csv")

In [ ]:
# loading dataset and building subgraph for simulation
def load_roadnet_edges(path):
    return pd.read_csv(
        path,
        sep=r"\s+",
        comment="#",
        header=None,
        names=["source", "target"],
        dtype={"source": int, "target": int},
    )


def build_subgraph(edge_df):
    G = nx.DiGraph()
    G.add_edges_from(edge_df.itertuples(index=False, name=None))
    return G


edge_df = load_roadnet_edges(DATA_PATH)
print(f"Loaded {len(edge_df):,} edges from {DATA_PATH}")

In [ ]:
EDGE_COUNTS = [50_000, 100_000, 200_000, 400_000,
               800_000, 1_600_000, 3_200_000, len(edge_df)]
REPEAT = 5
results = []

total_runs = len(EDGE_COUNTS) * REPEAT
with tqdm(total=total_runs, desc="RoadNet-CA experiments") as pbar:
    for edges_count in EDGE_COUNTS:
        subset = edge_df.iloc[:edges_count]
        for trial in range(REPEAT):
            G = build_subgraph(subset)
            start = time.perf_counter()
            Greedy_SOSP(G)
            end = time.perf_counter()

            results.append({
                "edges": edges_count,
                "nodes": G.number_of_nodes(),
                "runtime": end - start,
                "trial": trial + 1,
            })
            pbar.update(1)


results_df = pd.DataFrame(results)
results_df.head()

*"results" can be used further to analyze extensively about the simulated results*

*In the same way, we can do for Robust_MOSP and other given datasets*